# HINN Colab Training Pipeline
Full end-to-end: environment setup, HINN training, baselines, MOO, figures, download.

**Run cells in order.** After Cell 2, the repo is checked out. After Cell 5, weights are ready.

In [26]:
# Cell 1: Install uv and clone repo
!pip install -q uv
!git clone https://github.com/sattary/2601_chip_paper.git
%cd 2601_chip_paper

In [27]:
# Cell 2: Pull latest code and install all dependencies
!git pull
!uv sync  # Installs ALL deps from pyproject.toml including xgboost

In [28]:
# Cell 3: Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [33]:
!git pull

In [34]:
!ls

In [35]:
# Cell 4: Monotonicity ground truth audit (produces paper Section 4.1 data)
# Quick mode: --max-groups 5000 for a fast sample, remove flag for full audit
!uv run python cli.py analysis monotonicity --max-groups 5000

In [ ]:
# Cell 5: Train HINN (primary model, seed=42)
# For multi-seed reporting: re-run with --seed 0, 7, 123, 999
!uv run python cli.py train train --epochs 300 --batch-size 1024 --seed 42

In [ ]:
# Cell 6: Train baselines (XGBoost + Vanilla MLP) — same data split
!uv run python cli.py train baselines --epochs 300 --seed 42

In [ ]:
# Cell 7: Run MOO — extract Pareto front from trained HINN surrogate
!uv run python cli.py moo run

In [ ]:
# Cell 8: Generate all publication figures
!uv run python cli.py plot training-dynamics
!uv run python cli.py plot pareto
!uv run python cli.py plot monotonicity
!uv run python cli.py plot comparison

In [ ]:
# Cell 9: Zip all results (weights, scalers, CSVs, SVGs) and download
import shutil
import sys

# Include results/ subdirs: models/, figs/, data/
shutil.make_archive("hinn_results", "zip", "results")
print("Created hinn_results.zip in the current directory.")

if "google.colab" in sys.modules:
    try:
        from google.colab import files
        files.download("hinn_results.zip")
    except Exception as e:
        print(f"Could not auto-download (are you in VS Code?): {e}")
        print("Please download the zip manually from the file explorer.")
else:
    print("Not running in Colab web interface, skipping auto-download.")
    print("Please download the zip manually from the file explorer.")
